In [ ]:
import matplotlib.pyplot as plt
import mplhep as hep

plt.rcParams.update({
    key: value
    for key, value in hep.styles.CMS.items()
    if key.startswith("font.") and key != "font.size"
})
plt.rcParams["mathtext.fontset"] = "stix"

plt.rcParams["xaxis.labellocation"] = "right"
plt.rcParams["yaxis.labellocation"] = "top"

In [ ]:
import pandas as pd
import os

In [ ]:
import glob

#run_basename = '20261311_ParTauDETR_5classes'
run_basename = '20261111_ParTauDETR_4classes_balancedClasses'

obj_cls_trsh = 0.5
obj_cls_trsh_str = f'{obj_cls_trsh:.3f}'.replace('.', 'p')
df_dir = os.path.expanduser(f'~/tmp/{run_basename}-model_best-{obj_cls_trsh_str}thr')

df = pd.read_parquet(os.path.join(df_dir, 'z_test_predictions.parquet'))
score_columns = ['tau_tagging_score', *(
    column for column in df.columns if column.startswith('tau_id_prob_')
)]
background_paths = sorted(glob.glob(os.path.join(df_dir, '*_test_predictions.parquet')))
background_frames = {
    os.path.basename(path).removesuffix('_test_predictions.parquet'): pd.read_parquet(
        path, columns=['gen_jet_p4', 'reco_jet_p4', 'pred_tau_daughter_meson_classes', *score_columns],
    )
    for path in background_paths
    if os.path.basename(path) != 'z_test_predictions.parquet'
}
if not background_frames:
    raise FileNotFoundError(f'No background prediction files found in {df_dir}')

In [ ]:
df.head()

In [ ]:
pd.DataFrame({
    'sample': list(background_frames),
    'jets': [len(frame) for frame in background_frames.values()],
})

In [ ]:
import numpy as np
from IPython.display import display
from scipy.optimize import differential_evolution

class_columns = [column for column in df.columns if column.startswith('tau_id_prob_')]
signal_classes = ('hadronic_tau', 'tau_electron', 'tau_muon')
tau_score_columns = [f'tau_id_prob_{name}' for name in signal_classes]
background_score_columns = [column for column in class_columns if column not in tau_score_columns]
background_classes = [column.removeprefix('tau_id_prob_') for column in background_score_columns]
missing_signal_columns = set(tau_score_columns) - set(class_columns)
if missing_signal_columns:
    raise ValueError(f'Missing signal scores: {sorted(missing_signal_columns)}')
if not background_classes or set(background_classes) - {'background', 'leptonic_background', 'mm', 'bhabha'}:
    raise ValueError(f'Unexpected background classes: {background_classes}')

background_class_by_sample = {}
for sample, frame in background_frames.items():
    candidates = (sample, 'leptonic_background', 'background') if sample in ('mm', 'bhabha') else ('background',)
    name = next((name for name in candidates if name in background_classes), None)
    if name is None:
        raise ValueError(f'No background score available for sample {sample!r}')
    if frame.empty:
        raise ValueError(f'No {sample} jets available to fit tau-ID weights')
    background_class_by_sample[sample] = name

background_groups = {
    name: [sample for sample, assigned_class in background_class_by_sample.items() if assigned_class == name]
    for name in background_classes
}
background_class_frames = {
    name: pd.concat([background_frames[sample] for sample in samples], ignore_index=True)
    for name, samples in background_groups.items() if samples
}

objectives = ('signal_weighted_f1', 'hadronic_tau_f1', 'process_weighted_f1', 'signal_background_f1')
process_weights = {'signal': 1.0, 'mm': 1.0, 'bhabha': 3.0, 'qq': 20.0}
f1_signal_weight = 0.5
if not 0 <= f1_signal_weight <= 1:
    raise ValueError('f1_signal_weight must be between 0 and 1')

lepton_counts = df['gen_jet_tau_vis_daughter_pdgs'].map(
    lambda pdgs: (
        sum(abs(pdg) == 11 for pdg in pdgs),
        sum(abs(pdg) == 13 for pdg in pdgs),
    ) if isinstance(pdgs, (list, tuple, np.ndarray)) else (0, 0)
)
truth_masks = {
    'hadronic_tau': (df['gen_jet_tau_decaymode'] != -1)
        & lepton_counts.map(lambda counts: counts == (0, 0)),
    'tau_electron': lepton_counts.map(lambda counts: counts == (1, 0)),
    'tau_muon': lepton_counts.map(lambda counts: counts == (0, 1)),
}
signal_class_indices = [class_columns.index(column) for column in tau_score_columns]
anchor_index = class_columns.index('tau_id_prob_hadronic_tau')
signal_targets = np.full(len(df), -1)
for name, class_index in zip(signal_classes, signal_class_indices):
    signal_targets[truth_masks[name].to_numpy()] = class_index
signal_indices = np.flatnonzero(signal_targets >= 0)
signal_counts = np.bincount(signal_targets[signal_indices], minlength=len(class_columns))
if np.any(signal_counts[signal_class_indices] == 0):
    raise ValueError('All signal classes need truth examples to fit every objective')
signal_fractions = signal_counts / signal_counts.sum()

rng = np.random.default_rng(42)
indices = rng.choice(signal_indices, size=min(len(signal_indices), 30000), replace=False)
sampled_signal_targets = signal_targets[indices]
fit_score_parts = [df.iloc[indices][class_columns].to_numpy()]
fit_process_parts = [np.full(len(indices), 'signal')]
for sample, frame in background_frames.items():
    selected = rng.choice(len(frame), size=min(len(frame), 10000), replace=False)
    fit_score_parts.append(frame.iloc[selected][class_columns].to_numpy())
    fit_process_parts.append(np.full(len(selected), sample))
fit_scores = np.concatenate(fit_score_parts)
fit_targets = np.pad(sampled_signal_targets, (0, len(fit_scores) - len(indices)), constant_values=-1)
fit_processes = np.concatenate(fit_process_parts)
process_masks = {name: fit_processes == name for name in ('signal', *background_frames)}
active_process_weights = {name: process_weights.get(name, 1.0) for name in process_masks}
fit_process_weights = np.empty(len(fit_scores))
for name, mask in process_masks.items():
    fit_process_weights[mask] = active_process_weights[name] / np.count_nonzero(mask)
background_class_indices = {
    sample: class_columns.index(f'tau_id_prob_{name}')
    for sample, name in background_class_by_sample.items()
}
free_indices = [index for index in range(len(class_columns)) if index != anchor_index]


def class_weights(log_free_weights):
    weights = np.ones(len(class_columns))
    weights[free_indices] = np.exp(log_free_weights)
    return weights


def binary_f1(truth, predicted, weights):
    true_positives = weights[truth & predicted].sum()
    denominator = weights[truth].sum() + weights[predicted].sum()
    return 2 * true_positives / denominator if denominator else 0.0


def weighted_error(log_free_weights, objective):
    weights = class_weights(log_free_weights)
    if objective == 'signal_weighted_f1':
        predicted = (fit_scores[:len(indices)] * weights).argmax(axis=1)
        true_positives = np.bincount(sampled_signal_targets[predicted == sampled_signal_targets],
                                     minlength=len(class_columns))
        true_counts = np.bincount(sampled_signal_targets, minlength=len(class_columns))
        predicted_counts = np.bincount(predicted, minlength=len(class_columns))
        denominators = true_counts + predicted_counts
        f1_scores = np.divide(2 * true_positives, denominators,
                              out=np.zeros(len(class_columns)), where=denominators > 0)
        return -np.dot(signal_fractions, f1_scores)

    predicted = (fit_scores * weights).argmax(axis=1)
    if objective == 'hadronic_tau_f1':
        return -binary_f1(fit_targets == anchor_index, predicted == anchor_index, fit_process_weights)

    predicted_signal = np.isin(predicted, signal_class_indices)
    signal_f1 = binary_f1(process_masks['signal'], predicted_signal, fit_process_weights)
    background_efficiencies = {
        sample: np.mean(predicted[process_masks[sample]] == class_index)
        for sample, class_index in background_class_indices.items()
    }
    if objective == 'signal_background_f1':
        background_efficiency = np.average(
            list(background_efficiencies.values()),
            weights=[active_process_weights[sample] for sample in background_efficiencies],
        )
        return -(f1_signal_weight * signal_f1 + (1 - f1_signal_weight) * background_efficiency)
    process_scores = {'signal': signal_f1, **background_efficiencies}
    return -np.average(list(process_scores.values()),
                       weights=[active_process_weights[name] for name in process_scores])


weight_table = pd.DataFrame({'unit': np.ones(len(class_columns))}, index=class_columns)
weight_table.index.name = 'Score'
fit_summary = {}
unit_log_weights = np.zeros(len(free_indices))
for objective in objectives:
    fit = differential_evolution(
        weighted_error, bounds=[(-4, 4)] * len(free_indices), args=(objective,),
        seed=42, maxiter=60, popsize=8, polish=False, x0=unit_log_weights,
    )
    weight_table[objective] = class_weights(fit.x)
    fit_summary[objective] = {'Unit score': -weighted_error(unit_log_weights, objective), 'Fitted score': -fit.fun}

display(weight_table.style.format('{:.4f}'))
display(pd.DataFrame.from_dict(fit_summary, orient='index').rename_axis('Objective').style.format('{:.4f}'))

In [ ]:
selected_objective = 'unit'
if selected_objective not in weight_table.columns:
    raise ValueError(f'Choose one of {list(weight_table.columns)}')
tau_id_weights = weight_table[selected_objective].rename('weight').copy()
print(f'Using tau-ID weights: {selected_objective}')
display(tau_id_weights.to_frame().style.format('{:.4f}'))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

class_labels = {
    'hadronic_tau': 'Hadronic tau',
    'tau_electron': 'Tau to electron',
    'tau_muon': 'Tau to muon',
    'background': 'Background',
    'leptonic_background': 'Leptonic background',
    'mm': 'mm',
    'bhabha': 'Bhabha',
}


def class_margin(frame, score_column, weights):
    weighted_scores = frame[class_columns].to_numpy() * weights.loc[class_columns].to_numpy()
    class_index = class_columns.index(score_column)
    competitors = np.delete(weighted_scores, class_index, axis=1).max(axis=1)
    return weighted_scores[:, class_index] / np.maximum(competitors, 1e-12)


def roc_row():
    fig, axes = plt.subplots(
        1, len(weight_table.columns), figsize=(7 * len(weight_table.columns), 5),
        squeeze=False, sharey=True, constrained_layout=True,
    )
    return fig, axes[0]


def format_roc_axis(ax, xlabel, ylabel, objective):
    ax.set(xlabel=xlabel, ylabel=ylabel, yscale='log', xlim=(0, 1), title=objective)
    ax.legend()
    ax.xaxis.set_major_locator(MultipleLocator(0.1))
    ax.xaxis.set_minor_locator(MultipleLocator(0.05))
    ax.grid(True, which='both', axis='y', alpha=0.3)
    ax.grid(True, which='minor', axis='x', alpha=0.3)
    ax.grid(True, which='major', axis='x', alpha=0.5)


def plot_class_roc_row(class_name, primary_frame, comparisons):
    if primary_frame.empty:
        raise ValueError(f'No truth {class_name} jets available for the ROC curve')
    score_column = f'tau_id_prob_{class_name}'
    label = class_labels[class_name]
    fig, axes = roc_row()
    fig.suptitle(f'{label} identification performance')
    for ax, (objective, weights) in zip(axes, weight_table.items()):
        primary_scores = class_margin(primary_frame, score_column, weights)
        cuts = np.unique(np.r_[np.percentile(primary_scores, np.linspace(0, 100, 100)), 1.0])
        efficiency = [np.mean(primary_scores >= cut) for cut in cuts]
        for frame, comparison_label, style in comparisons:
            if frame.empty:
                continue
            scores = class_margin(frame, score_column, weights)
            misid = [np.mean(scores >= cut) for cut in cuts]
            ax.plot(efficiency, misid, label=comparison_label, **style)
        format_roc_axis(ax, f'{label} identification efficiency',
                        f'{label} misidentification rate', objective)
    plt.show()
    plt.close(fig)


signal_class_frames = {class_name: df.loc[mask, class_columns] for class_name, mask in truth_masks.items()}
background_comparisons = [(frame, sample, {}) for sample, frame in background_frames.items()]
if len(background_frames) > 1:
    pooled_background = pd.concat([frame[class_columns] for frame in background_frames.values()], ignore_index=True)
    background_comparisons.append((pooled_background, 'All backgrounds', {'color': 'black', 'linestyle': '--'}))

for class_name, primary_frame in signal_class_frames.items():
    comparisons = background_comparisons + [
        (frame, f'{class_labels[other_class]} (tau mis-ID)', {'linestyle': ':'})
        for other_class, frame in signal_class_frames.items() if other_class != class_name
    ]
    plot_class_roc_row(class_name, primary_frame, comparisons)

if len(background_classes) > 1:
    signal_mask = pd.concat(list(truth_masks.values()), axis=1).any(axis=1)
    signal_comparisons = [(df.loc[signal_mask, class_columns], 'Signal', {})] + [
        (frame, class_labels[class_name], {'linestyle': ':'})
        for class_name, frame in signal_class_frames.items()
    ]
    for class_name, primary_frame in background_class_frames.items():
        comparisons = signal_comparisons + [
            (frame, class_labels[other_class], {})
            for other_class, frame in background_class_frames.items() if other_class != class_name
        ]
        plot_class_roc_row(class_name, primary_frame, comparisons)

In [ ]:
from IPython.display import HTML, display

working_points = np.array([80, 90, 95, 97.5, 99, 99.5])


def signal_background_ratio(frame, weights):
    tau_scores = (
        frame[tau_score_columns].to_numpy(dtype=float)
        * weights.loc[tau_score_columns].to_numpy(dtype=float)
    ).sum(axis=1)
    background_scores = (
        frame[background_score_columns].to_numpy(dtype=float)
        * weights.loc[background_score_columns].to_numpy(dtype=float)
    ).sum(axis=1)
    if not np.all(np.isfinite(tau_scores) & np.isfinite(background_scores)):
        raise ValueError('Tau and background scores must be finite')
    if np.any(tau_scores < 0) or np.any(background_scores < 0):
        raise ValueError('Tau and background scores must be nonnegative')
    return tau_scores / np.maximum(background_scores, np.finfo(float).tiny)


def plot_signal_background_roc(ax, objective, weights, signal_frame, xlabel):
    sample_ratios = {
        sample: signal_background_ratio(frame, weights)
        for sample, frame in {'z': signal_frame, **background_frames}.items()
    }
    if not sample_ratios['z'].size:
        raise ValueError(f'No signal jets in sample z for {xlabel}')

    sorted_signal_ratios = np.sort(sample_ratios['z'])
    required_counts = np.ceil(working_points / 100 * len(sorted_signal_ratios)).astype(int)
    thresholds = np.nextafter(sorted_signal_ratios[-required_counts], -np.inf)
    table = pd.DataFrame({
        f'{efficiency:g}%': {
            sample: 100 * np.mean(ratios > threshold) if ratios.size else np.nan
            for sample, ratios in sample_ratios.items()
        }
        for efficiency, threshold in zip(working_points, thresholds)
    })
    table.index.name = 'Sample'
    table.columns.name = 'Target z efficiency'

    for sample, ratios in sample_ratios.items():
        if sample == 'z' or not ratios.size:
            continue
        sorted_background_ratios = np.sort(ratios)
        roc_thresholds = np.r_[
            np.inf,
            np.unique(np.concatenate((sorted_signal_ratios, sorted_background_ratios)))[::-1],
            -np.inf,
        ]
        roc_signal_efficiency = (
            1 - np.searchsorted(sorted_signal_ratios, roc_thresholds, side='right')
            / len(sorted_signal_ratios)
        )
        roc_misidentification = (
            1 - np.searchsorted(sorted_background_ratios, roc_thresholds, side='right')
            / len(sorted_background_ratios)
        )
        curve, = ax.plot(roc_signal_efficiency, roc_misidentification, label=sample)
        ax.scatter(table.loc['z'] / 100, table.loc[sample] / 100,
                   color=curve.get_color(), s=25, zorder=3)
    format_roc_axis(ax, xlabel, 'Background misidentification rate', objective)
    return thresholds, table


def plot_signal_background_roc_row(signal_frame, title, xlabel):
    fig, axes = roc_row()
    fig.suptitle(title)
    thresholds_by_objective = {}
    tables = {}
    for ax, (objective, weights) in zip(axes, weight_table.items()):
        thresholds, table = plot_signal_background_roc(ax, objective, weights, signal_frame, xlabel)
        thresholds_by_objective[objective] = thresholds
        tables[objective] = table
    plt.show()
    plt.close(fig)
    return thresholds_by_objective, tables


working_point_thresholds_by_objective, working_point_tables = plot_signal_background_roc_row(
    df, 'Summed tau identification performance: all z jets', 'Signal identification efficiency in z',
)
hadronic_working_point_thresholds_by_objective, hadronic_working_point_tables = plot_signal_background_roc_row(
    df.loc[truth_masks['hadronic_tau'], class_columns],
    'Summed tau identification performance: truth hadronic taus in z',
    'Hadronic tau identification efficiency in z',
)

display(HTML('<p>Passing jets [%]: z signal efficiency; other samples background misidentification</p>'))
for objective, table in working_point_tables.items():
    display(HTML(f'<h4>Working points: {objective}</h4>' + table.to_html(float_format=lambda value: f'{value:.4f}')))

working_point_thresholds = working_point_thresholds_by_objective[selected_objective]
working_point_table = working_point_tables[selected_objective]

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

explicit_dms = [0, 1, 2, 10, 11, 12]
background_samples = {'qq': 'qqbar', 'mm': 'mm', 'bhabha': 'bhabha'}
if set(background_frames) != set(background_samples):
    raise ValueError(f'Expected background samples {list(background_samples)}, got {list(background_frames)}')
background_labels = [class_labels[name] for name in background_classes]
hadronic_dm_labels = explicit_dms + ["Rare"]
signal_labels = hadronic_dm_labels + ["Muon", "Electron"]
labels = signal_labels + list(background_samples.values())
predicted_labels = signal_labels + ["Invalid"] + background_labels


def normalize_truth_dm(dm, lepton_counts):
    if lepton_counts == (1, 0):
        return "Electron"
    if lepton_counts == (0, 1):
        return "Muon"
    return dm if dm in explicit_dms else "Rare"


def predict_dm(class_array, tau_id_class):
    if tau_id_class == "tau_electron":
        return "Electron"
    if tau_id_class == "tau_muon":
        return "Muon"
    if tau_id_class in background_classes:
        return class_labels[tau_id_class]
    if tau_id_class != "hadronic_tau":
        raise ValueError(f"Unexpected tau-ID class: {tau_id_class}")
    if not isinstance(class_array, (list, tuple, np.ndarray)):
        return "Invalid"
    n_charged = sum(class_value == 0 for class_value in class_array)
    n_neutral = sum(class_value == 1 for class_value in class_array)
    if n_charged % 2 == 0 or n_neutral > 4:
        return "Invalid"
    dm = 5 * (n_charged - 1) + n_neutral
    return dm if dm in explicit_dms else "Rare"


lepton_counts = df["gen_jet_tau_vis_daughter_pdgs"].map(
    lambda pdgs: (
        sum(abs(pdg) == 11 for pdg in pdgs),
        sum(abs(pdg) == 13 for pdg in pdgs),
    ) if isinstance(pdgs, (list, tuple, np.ndarray)) else (0, 0)
)
signal_true_dm = pd.Series(
    [normalize_truth_dm(dm, counts) for dm, counts in zip(df["gen_jet_tau_decaymode"], lepton_counts)]
)
all_frames = [df, *(background_frames[sample] for sample in background_samples)]
all_predictions = pd.concat(all_frames, ignore_index=True)
true_dm = pd.concat([
    signal_true_dm,
    *(pd.Series(label, index=range(len(background_frames[sample])))
      for sample, label in background_samples.items()),
], ignore_index=True)


def plot_confusion_pair(objective, weights):
    tau_ids = (all_predictions[class_columns].mul(weights).idxmax(axis=1)
               .astype(str).str.removeprefix("tau_id_prob_"))
    decay_modes = pd.Series([
        predict_dm(classes, tau_id)
        for classes, tau_id in zip(all_predictions["pred_tau_daughter_meson_classes"], tau_ids)
    ])
    counts = pd.crosstab(decay_modes, true_dm).reindex(index=predicted_labels, columns=labels, fill_value=0)
    truth_totals = counts.sum(axis=0).replace(0, np.nan)
    efficiency = counts.div(truth_totals, axis=1).fillna(0)
    signal_counts = counts.loc[:, signal_labels]
    precision = signal_counts.div(signal_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    background_columns = list(background_samples.values())
    hadronic_counts = counts.loc[hadronic_dm_labels, background_columns].sum()
    hadronic_efficiency = hadronic_counts.div(truth_totals.loc[background_columns])

    fig, axes = plt.subplots(
        1, 2, figsize=(max(16, 1.6 * len(labels)), max(7, 0.75 * (len(predicted_labels) + 1))),
        constrained_layout=True, gridspec_kw={"wspace": 0.16},
    )
    fig.suptitle(f"Decay-mode confusion matrices: {objective}")
    cmap = plt.get_cmap("Blues").with_extremes(bad="0.9")

    for ax, matrix, matrix_counts, title in zip(
        axes,
        [efficiency, precision],
        [counts, signal_counts],
        ["Efficiency: P(predicted | true)", "Precision: P(true | predicted, signal)"],
    ):
        matrix = matrix.iloc[::-1]
        display_counts = matrix_counts.iloc[::-1]
        if ax is axes[0]:
            summary = hadronic_efficiency.reindex(matrix.columns).to_frame(name="Hadronic tau").T
            summary_counts = hadronic_counts.reindex(matrix.columns).to_frame(name="Hadronic tau").T
            matrix = pd.concat([summary, matrix])
            display_counts = pd.concat([summary_counts, display_counts])
        image = ax.imshow(matrix, vmin=0, vmax=1, cmap=cmap, origin="upper")
        ax.set(
            title=title,
            xticks=range(len(matrix.columns)),
            xticklabels=matrix.columns,
            yticks=range(len(matrix.index)),
            yticklabels=matrix.index,
        )
        ax.set_xlabel("True decay mode or process", loc="right", fontsize=14)
        ax.set_ylabel("Predicted decay mode or process", loc="top", fontsize=14)
        ax.tick_params(axis="x", rotation=45)
        if ax is axes[0]:
            ax.axhline(0.5, color="black", linewidth=1.3)
            for column_boundary in range(len(signal_labels), len(labels)):
                ax.axvline(column_boundary - 0.5, color="black",
                           linewidth=1.3 if column_boundary == len(signal_labels) else 0.7)
        for row_index in range(matrix.shape[0]):
            for column_index in range(matrix.shape[1]):
                cell_value = matrix.iloc[row_index, column_index]
                if pd.isna(cell_value):
                    ax.text(column_index, row_index, "n/a", ha="center", va="center", fontsize=10)
                    continue
                count = display_counts.iloc[row_index, column_index]
                uncertainty = cell_value / np.sqrt(count) if count else 0.0
                color = "white" if cell_value > 0.5 else "black"
                for text, offset, font_size in (
                    (f"{cell_value:.3f}", 6, 10),
                    (f"+/- {uncertainty:.3f}", -7, 7),
                ):
                    ax.annotate(
                        text,
                        (column_index, row_index),
                        xytext=(0, offset),
                        textcoords="offset points",
                        ha="center",
                        va="center",
                        fontsize=font_size,
                        color=color,
                    )
        colorbar_ax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.08)
        fig.colorbar(image, cax=colorbar_ax, label="Fraction")

    plt.show()
    plt.close(fig)
    return tau_ids, decay_modes, counts, efficiency, signal_counts, precision


for objective, weights in weight_table.items():
    results = plot_confusion_pair(objective, weights)
    if objective == selected_objective:
        (predicted_tau_id, predicted_dm, confusion_counts, confusion_efficiency,
         precision_counts, confusion_precision) = results

In [ ]:
# Energy spectrum comparison: jets tagged as qqbar vs hadronic tau
# Extract jet energies from reconstructed jet 4-momenta
# all_predictions already contains the concatenated signal and background data
def extract_energy(p4):
    """Extract energy from a 4-momentum object (dict, LorentzVector, or array-like)."""
    if isinstance(p4, dict):
        # Try common energy key names
        for key in ("energy", "E", "t", "e"):
            if key in p4:
                return p4[key]
    # Try attribute access
    if hasattr(p4, 'E'):
        return p4.E
    if hasattr(p4, 'energy'):
        return p4.energy
    # Try as array/list (first element is often energy or time component)
    if hasattr(p4, '__getitem__') and len(p4) > 0:
        return p4[0]
    # Fallback: return as-is
    return p4

reco_jet_energies = all_predictions["reco_jet_p4"].map(extract_energy)

# Create masks for jets tagged as qqbar and hadronic_tau based on tau ID score
qqbar_mask = predicted_tau_id == "background"
hadronic_tau_mask = predicted_tau_id == "hadronic_tau"

# Extract energies for each category
qqbar_energies = reco_jet_energies[qqbar_mask]
hadronic_tau_energies = reco_jet_energies[hadronic_tau_mask]

# Define binning: 0 to 45 GeV with bin width of 0.5 GeV
bins = np.arange(0, 45.5, 0.5)

# Create the plot with normalized histograms
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(
    qqbar_energies,
    bins=bins,
    density=True,  # Normalize to unit area
    alpha=0.6,
    label='Tagged as qqbar',
    color='tab:red',
    histtype='stepfilled',
    linewidth=1.5
)

ax.hist(
    hadronic_tau_energies,
    bins=bins,
    density=True,  # Normalize to unit area
    alpha=0.6,
    label='Tagged as hadronic tau',
    color='tab:blue',
    histtype='stepfilled',
    linewidth=1.5
)

ax.set_xlabel('Reconstructed jet energy [GeV]', fontsize=14)
ax.set_ylabel('Normalised distribution', fontsize=14)
ax.set_title('Energy spectrum of hadronic jets by tau ID classification', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim(0, 45)

plt.tight_layout()
plt.show()
plt.close()

# Print statistics
print(f"Jets tagged as qqbar: {qqbar_mask.sum()}")
print(f"Jets tagged as hadronic tau: {hadronic_tau_mask.sum()}")
print(f"Mean energy (qqbar): {qqbar_energies.mean():.2f} GeV")
print(f"Mean energy (hadronic tau): {hadronic_tau_energies.mean():.2f} GeV")
print(f"Median energy (qqbar): {qqbar_energies.median():.2f} GeV")
print(f"Median energy (hadronic tau): {hadronic_tau_energies.median():.2f} GeV")

In [ ]:
reco_charged_counts = df["reco_cand_charges"].map(
    lambda charges: np.count_nonzero(np.abs(charges) == 1)
)
predicted_charged_counts = df["pred_tau_daughter_meson_classes"].map(
    lambda classes: sum(class_value == 0 for class_value in classes)
    if isinstance(classes, (list, tuple, np.ndarray)) else 0
)
charged_counts = pd.crosstab(predicted_charged_counts, reco_charged_counts).reindex(
    index=range(6), columns=range(6), fill_value=0
)
charged_matrix = charged_counts.div(
    charged_counts.sum(axis=0).replace(0, np.nan), axis=1
).fillna(0)

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
image = ax.imshow(charged_matrix, origin="lower", vmin=0, vmax=1, cmap="Blues")
ax.set(
    xticks=range(6), yticks=range(6),
    xlabel="Number of reconstructed charged constituents",
    ylabel="Number of predicted charged constituents",
)
for row_index in range(6):
    for column_index in range(6):
        value = charged_matrix.iloc[row_index, column_index]
        ax.text(column_index, row_index, f"{value:.3f}", ha="center", va="center",
                color="white" if value > 0.5 else "black")
fig.colorbar(image, ax=ax, label="Fraction of reconstructed count")
plt.show()
plt.close()

In [ ]:
charged_cut = pd.concat([
    predicted_charged_counts.le(reco_charged_counts),
    pd.Series(True, index=predicted_dm.index[len(df):]),
], ignore_index=True)

restricted_counts = pd.crosstab(predicted_dm[charged_cut], true_dm[charged_cut]).reindex(
    index=predicted_labels, columns=labels, fill_value=0,
)
restricted_efficiency = restricted_counts.div(
    confusion_counts.sum(axis=0).replace(0, np.nan), axis=1
).fillna(0)
restricted_precision_counts = restricted_counts.loc[:, signal_labels]
restricted_precision = restricted_precision_counts.div(
    restricted_precision_counts.sum(axis=1).replace(0, np.nan), axis=0
).fillna(0)

efficiency_change = restricted_efficiency - confusion_efficiency
precision_change = restricted_precision - confusion_precision

fig, axes = plt.subplots(
    1, 2, figsize=(max(16, 1.6 * len(labels)), max(7, 0.75 * len(predicted_labels))),
    constrained_layout=True, gridspec_kw={"wspace": 0.16},
)
for ax, change, title in zip(
    axes,
    [efficiency_change, precision_change],
    ["Change in efficiency", "Change in precision"],
):
    change = change.iloc[::-1]
    change_values = change.to_numpy(dtype=float)
    limit = max(np.abs(change_values).max(), 1e-12)
    image = ax.imshow(change_values, vmin=-limit, vmax=limit, cmap="RdBu_r")
    ax.set(
        title=title,
        xticks=range(len(change.columns)), xticklabels=change.columns,
        yticks=range(len(change.index)), yticklabels=change.index,
    )
    ax.set_xlabel("True decay mode or process", loc="right", fontsize=14)
    ax.set_ylabel("Predicted decay mode or process", loc="top", fontsize=14)
    ax.tick_params(axis="x", rotation=45)
    for row_index in range(change.shape[0]):
        for column_index in range(change.shape[1]):
            value = change_values[row_index, column_index]
            ax.text(column_index, row_index, f"{value:.3f}",
                    ha="center", va="center", fontsize=9,
                    color="white" if abs(value) > limit / 2 else "black")
    colorbar_ax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.08)
    fig.colorbar(image, cax=colorbar_ax, label="Change in fraction")
plt.show()
plt.close()

In [ ]:
dm10_mask = pd.Series(
    signal_true_dm.eq(10).to_numpy() & predicted_dm.iloc[:len(df)].eq(10).to_numpy(),
    index=df.index,
)

In [ ]:
dm10_predicted_charge_sum = df.loc[dm10_mask, "pred_tau_daughter_charges"].map(np.sum)
dm10_reco_charge_sum = df.loc[dm10_mask, "reco_cand_charges"].map(np.sum)
minimum_charge = min(dm10_predicted_charge_sum.min(), dm10_reco_charge_sum.min())
maximum_charge = max(dm10_predicted_charge_sum.max(), dm10_reco_charge_sum.max())
charge_bins = np.arange(minimum_charge - 0.5, maximum_charge + 1.5)

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.hist(dm10_reco_charge_sum, bins=charge_bins, density=True, alpha=0.6,
        label="Reconstructed", color="tab:blue", edgecolor="white")
ax.hist(dm10_predicted_charge_sum, bins=charge_bins, density=True, alpha=0.6,
        label="Predicted", color="tab:orange", edgecolor="white")
ax.set(
    title="Generator-level and predicted DM 10",
    xlabel="Sum of constituent charges",
    ylabel="Fraction of jets",
    xticks=np.arange(minimum_charge, maximum_charge + 1),
)
ax.set_yscale('log')
ax.legend()
plt.show()
plt.close()

In [ ]:
charge_values = np.arange(-2, 3)
charge_sum_counts = pd.crosstab(dm10_predicted_charge_sum, dm10_reco_charge_sum).reindex(
    index=charge_values, columns=charge_values, fill_value=0
)
charge_sum_matrix = charge_sum_counts.div(
    charge_sum_counts.sum(axis=1).replace(0, np.nan), axis=0
).fillna(0)

fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
charge_sum_values = charge_sum_matrix.to_numpy(dtype=float)
image = ax.imshow(charge_sum_values, origin="lower", vmin=0, vmax=1, cmap="Blues")
ax.set(
    title="Generator-level and predicted DM 10",
    xticks=range(len(charge_values)), xticklabels=charge_values,
    yticks=range(len(charge_values)), yticklabels=charge_values,
    xlabel="Sum of reconstructed constituent charges",
    ylabel="Sum of predicted constituent charges",
)
for row_index in range(len(charge_values)):
    for column_index in range(len(charge_values)):
        value = charge_sum_values[row_index, column_index]
        ax.text(column_index, row_index, f"{value:.3f}", ha="center", va="center",
                color="white" if value > 0.5 else "black")
fig.colorbar(image, ax=ax, label="Fraction at predicted charge sum")
plt.show()
plt.close()

In [ ]:
for predicted_charge in (1, -1):
    selected = dm10_reco_charge_sum.eq(0) & dm10_predicted_charge_sum.eq(predicted_charge)
    pdg_lists = df.loc[
        selected[selected].index,
        ["reco_cand_pdgs", "reco_cand_charges", "gen_jet_tau_vis_daughter_pdgs"],
    ].head(10)
    print(f"Reconstructed charge sum 0, predicted charge sum {predicted_charge:+d}:")
    for jet_index, (reco_pdgs, reco_charges, gen_pdgs) in zip(
        pdg_lists.index, pdg_lists.itertuples(index=False, name=None)
    ):
        signed_reco_pdgs = [
            int(pdg * (charge if charge != 0 else 1))
            for pdg, charge in zip(reco_pdgs, reco_charges)
        ]
        print(f"  Jet {jet_index}: reco {signed_reco_pdgs}"
              f" | gen {[int(pdg) for pdg in gen_pdgs]}")
    print()

In [ ]:
target_pdgs = sorted([211, -211, 130])
for predicted_charge in (1, -1):
    selected = dm10_reco_charge_sum.eq(0) & dm10_predicted_charge_sum.eq(predicted_charge)
    reco_particles = df.loc[selected[selected].index, ["reco_cand_pdgs", "reco_cand_charges"]]
    matches = sum(
        sorted(
            int(pdg * (charge if charge != 0 else 1))
            for pdg, charge in zip(reco_pdgs, reco_charges)
        ) == target_pdgs
        for reco_pdgs, reco_charges in reco_particles.itertuples(index=False, name=None)
    )
    total = len(reco_particles)
    fraction = matches / total if total else float("nan")
    print(f"Predicted charge sum {predicted_charge:+d}: {matches}/{total} = {fraction:.3f}")

In [ ]:
pdg130_kinematics = {1: ([], [], []), -1: ([], [], [])}
for predicted_charge, (energies, thetas, transverse_momenta) in pdg130_kinematics.items():
    selected = dm10_reco_charge_sum.eq(0) & dm10_predicted_charge_sum.eq(predicted_charge)
    particles = df.loc[
        selected[selected].index, ["reco_cand_pdgs", "reco_cand_charges", "reco_cand_p4s"]
    ]
    for reco_pdgs, reco_charges, reco_p4s in particles.itertuples(index=False, name=None):
        for pdg, charge, p4 in zip(reco_pdgs, reco_charges, reco_p4s):
            if int(pdg * (charge if charge != 0 else 1)) == 130:
                energies.append(float(next(p4[field] for field in ("energy", "t", "E", "e") if field in p4)))
                thetas.append(float(np.degrees(2 * np.arctan(np.exp(-p4["eta"])))))
                transverse_momenta.append(float(p4["pt"] if "pt" in p4 else p4["rho"]))

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for ax, position, edges, xlabel in (
    (axes[0], 0, np.linspace(0, 45, 31), "Reconstructed PDG 130 energy [GeV]"),
    (axes[1], 1, np.linspace(10, 170, 33), "Reconstructed PDG 130 theta [degrees]"),
    (axes[2], 2, np.linspace(0, 45, 31), "Reconstructed PDG 130 pT [GeV]"),
):
    for predicted_charge, color in ((1, "tab:orange"), (-1, "tab:blue")):
        values = [value for value in pdg130_kinematics[predicted_charge][position]
                  if edges[0] <= value <= edges[-1]]
        if values:
            ax.hist(values, bins=edges, density=True, histtype="step", linewidth=2,
                    label=f"Predicted charge {predicted_charge:+d} (n={len(values)})", color=color)
    ax.set(xlabel=xlabel, ylabel="Probability density", xlim=(edges[0], edges[-1]))
    if ax.has_data():
        ax.legend()
fig.suptitle("Generator-level and predicted DM 10, reconstructed charge sum 0")
plt.show()
plt.close()

In [ ]:
selected = dm10_reco_charge_sum.eq(0) & dm10_predicted_charge_sum.isin((-1, 1))
reco_particles = df.loc[selected[selected].index, ["reco_cand_pdgs", "reco_cand_charges"]]
matching_indices = [
    jet_index
    for jet_index, (reco_pdgs, reco_charges) in zip(
        reco_particles.index, reco_particles.itertuples(index=False, name=None)
    )
    if sorted(
        int(pdg * (charge if charge != 0 else 1))
        for pdg, charge in zip(reco_pdgs, reco_charges)
    ) == target_pdgs
]
charge_confusion_counts = pd.crosstab(
    dm10_predicted_charge_sum.loc[matching_indices],
    df.loc[matching_indices, "gen_jet_tau_charge"],
).reindex(index=[-1, 1], columns=[-1, 1], fill_value=0)
charge_confusion = charge_confusion_counts.div(
    charge_confusion_counts.sum(axis=0).replace(0, np.nan), axis=1
).fillna(0)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
charge_fractions = charge_confusion.to_numpy(dtype=float)
image = ax.imshow(charge_fractions, origin="lower", vmin=0, vmax=1, cmap="Blues")
ax.set(
    title="DM 10, reconstructed PDGs [211, -211, 130]",
    xticks=range(2), xticklabels=[-1, 1],
    yticks=range(2), yticklabels=[-1, 1],
    xlabel="Generator-level charge", ylabel="Predicted charge",
)
for row_index in range(2):
    for column_index in range(2):
        value = charge_fractions[row_index, column_index]
        ax.text(column_index, row_index, f"{value:.3f}", ha="center", va="center",
                color="white" if value > 0.5 else "black")
fig.colorbar(image, ax=ax, label="Fraction at generator-level charge")
plt.show()
plt.close()

In [ ]:
selected = dm10_mask & dm10_reco_charge_sum.isin((-1, 1)) & dm10_predicted_charge_sum.isin((-1, 1))
subset = df.loc[selected[selected].index]

reco_pdgs_list = subset["reco_cand_pdgs"].tolist()
reco_charges_list = subset["reco_cand_charges"].tolist()
reco_p4s_list = subset["reco_cand_p4s"].tolist()
gen_daughter_p4s_list = subset["gen_jet_tau_vis_daughter_p4s"].tolist()
gen_daughter_charges_list = subset["gen_jet_tau_vis_daughter_charges"].tolist()
gen_charges = subset["gen_jet_tau_charge"].to_numpy()
predicted_charge_sums = dm10_predicted_charge_sum.loc[selected[selected].index].to_numpy()

results = {1: {"masses_closest": [], "masses_furthest": [], "charge_diffs": [], "dR_gen": [], "q_dphi": [], "q_dphi2": [], "neutral_energies": [], "q_diff": []}, 
           -1: {"masses_closest": [], "masses_furthest": [], "charge_diffs": [], "dR_gen": [], "q_dphi": [], "q_dphi2": [], "neutral_energies": [], "q_diff": []}}

for reco_pdgs, reco_charges, reco_p4s, gen_daughter_p4s, gen_daughter_charges, gen_charge, pred_charge in zip(
    reco_pdgs_list, reco_charges_list, reco_p4s_list, gen_daughter_p4s_list, gen_daughter_charges_list, gen_charges, predicted_charge_sums
):
    neutral_indices = [i for i, pdg in enumerate(reco_pdgs) if int(pdg * (reco_charges[i] if reco_charges[i] != 0 else 1)) == 130]
    charged_indices = [i for i, pdg in enumerate(reco_pdgs) if abs(int(pdg * (reco_charges[i] if reco_charges[i] != 0 else 1))) == 211]
    
    if len(neutral_indices) != 1 or len(charged_indices) != 2:
        continue
    
    neutral_idx = neutral_indices[0]
    neutral_p4 = reco_p4s[neutral_idx]
    neutral_eta = float(neutral_p4.get("eta", 0))
    neutral_phi = float(neutral_p4.get("phi", 0))
    neutral_energy = float(neutral_p4.get("energy", 0))
    results[pred_charge]["neutral_energies"].append(neutral_energy)
    
    charged_p4s = [reco_p4s[i] for i in charged_indices]
    charged_charges = [reco_charges[i] for i in charged_indices]
    
    distances = []
    for charged_p4, charged_charge in zip(charged_p4s, charged_charges):
        charged_eta = float(charged_p4.get("eta", 0))
        charged_phi = float(charged_p4.get("phi", 0))
        delta_eta = charged_eta - neutral_eta
        delta_phi = charged_phi - neutral_phi
        distance = np.sqrt(delta_eta**2 + delta_phi**2)
        distances.append((distance, charged_p4, charged_charge))
    
    distances.sort(key=lambda x: x[0])
    closest_charged_p4 = distances[0][1]
    closest_charged_charge = distances[0][2]
    furthest_charged_p4 = distances[1][1]
    
    gen_charges_list = list(gen_daughter_charges)
    charge_sum = sum(gen_charges_list)
    matching_gen_indices = [i for i, ch in enumerate(gen_charges_list) if ch == charge_sum]
    opposite_gen_indices = [i for i, ch in enumerate(gen_charges_list) if ch == -charge_sum]

    min_e_factor = float('inf')
    q_estimated = 0
    for charged_p4, charged_charge in zip(charged_p4s, charged_charges):
        energy = float(charged_p4.get("energy", 0))
        charged_eta = float(charged_p4.get("eta", 0))
        charged_phi = float(charged_p4.get("phi", 0))

        delta_eta = charged_eta - neutral_eta
        delta_phi = charged_phi - neutral_phi

        cos_theta = np.cosh(delta_eta) - np.cos(delta_phi)
        cos_theta = cos_theta / (np.cosh(charged_eta) * np.cosh(neutral_eta))
        cos_theta = np.clip(cos_theta, -1.0, 1.0)
        theta = np.arccos(cos_theta)
        
        e_factor = energy * (1 - np.cos(theta))
        if e_factor < min_e_factor:
            min_e_factor = e_factor
            q_estimated = charged_charge
    
    q_diff = q_estimated - charge_sum
    results[pred_charge]["q_diff"].append(q_diff)
    
    if not matching_gen_indices:
        continue
    
    gen_p4s_list = list(gen_daughter_p4s)
    gen_distances = []
    for idx in matching_gen_indices:
        gen_p4 = gen_p4s_list[idx]
        gen_eta = float(gen_p4.get("eta", 0))
        gen_phi = float(gen_p4.get("phi", 0))
        d_eta = neutral_eta - gen_eta
        d_phi = neutral_phi - gen_phi
        d_phi_wrapped = (d_phi + np.pi) % (2 * np.pi) - np.pi
        dR = np.sqrt(d_eta**2 + d_phi**2)
        gen_distances.append((dR, gen_phi, gen_p4))
    
    if not gen_distances:
        continue
    
    gen_distances.sort(key=lambda x: x[0])
    best_dR = gen_distances[0][0]
    best_gen_phi = gen_distances[0][1]
    best_gen_p4 = gen_distances[0][2]
    
    d_phi = neutral_phi - best_gen_phi
    d_phi_wrapped = (d_phi + np.pi) % (2 * np.pi) - np.pi
    q_dphi = charge_sum * d_phi_wrapped

    assert(opposite_gen_indices)
    gen2_phi = float(gen_p4s_list[opposite_gen_indices[0]].get("phi", 0))
    d_phi_gen = best_gen_phi - gen2_phi
    d_phi_gen_wrapped = (d_phi_gen + np.pi) % (2 * np.pi) - np.pi
    q_dphi2 = charge_sum * d_phi_gen_wrapped
    results[pred_charge]["q_dphi2"].append(q_dphi2)
    
    results[pred_charge]["dR_gen"].append(best_dR)
    results[pred_charge]["q_dphi"].append(q_dphi)
    
    def add_p4(p4_a, p4_b):
        import math
        pt_a, eta_a, phi_a = p4_a["pt"], p4_a["eta"], p4_a["phi"]
        pt_b, eta_b, phi_b = p4_b["pt"], p4_b["eta"], p4_b["phi"]
        px_a = pt_a * math.cos(phi_a)
        py_a = pt_a * math.sin(phi_a)
        pz_a = pt_a * math.sinh(eta_a)
        px_b = pt_b * math.cos(phi_b)
        py_b = pt_b * math.sin(phi_b)
        pz_b = pt_b * math.sinh(eta_b)
        E_a = p4_a.get("energy", math.sqrt(pt_a**2 * math.cosh(eta_a)**2))
        E_b = p4_b.get("energy", math.sqrt(pt_b**2 * math.cosh(eta_b)**2))
        return {
            "px": px_a + px_b,
            "py": py_a + py_b,
            "pz": pz_a + pz_b,
            "E": E_a + E_b,
        }
    
    def mass(p4):
        return float(np.sqrt(max(0, p4["E"]**2 - p4["px"]**2 - p4["py"]**2 - p4["pz"]**2)))
    
    combined_p4_closest = add_p4(neutral_p4, closest_charged_p4)
    pair_mass_closest = mass(combined_p4_closest)
    combined_p4_furthest = add_p4(neutral_p4, furthest_charged_p4)
    pair_mass_furthest = mass(combined_p4_furthest)
    charge_diff = closest_charged_charge - gen_charge
    
    results[pred_charge]["masses_closest"].append(pair_mass_closest)
    results[pred_charge]["masses_furthest"].append(pair_mass_furthest)
    results[pred_charge]["charge_diffs"].append(charge_diff)

fig, axes = plt.subplots(1, 4, figsize=(18, 5), constrained_layout=True)

ax1 = axes[0]
for predicted_charge, color in ((1, "tab:orange"), (-1, "tab:blue")):
    masses = results[predicted_charge]["masses_closest"]
    if masses:
        bins = np.linspace(0, max(masses) * 1.1, 30)
        ax1.hist(masses, bins=bins, density=True, histtype="step", linewidth=2,
                 label=f"Predicted charge {predicted_charge:+d} (n={len(masses)})", color=color)
ax1.set(xlabel="Mass of (π⁰ + closest π±) pair [GeV]", ylabel="Probability density")
ax1.legend()

ax2 = axes[1]
for predicted_charge, color in ((1, "tab:orange"), (-1, "tab:blue")):
    masses = results[predicted_charge]["masses_furthest"]
    if masses:
        bins = np.linspace(0, max(masses) * 1.1, 30)
        ax2.hist(masses, bins=bins, density=True, histtype="step", linewidth=2,
                 label=f"Predicted charge {predicted_charge:+d} (n={len(masses)})", color=color)
ax2.set(xlabel="Mass of (π⁰ + furthest π±) pair [GeV]", ylabel="Probability density")
ax2.legend()

ax3 = axes[2]
for predicted_charge, color in ((1, "tab:orange"), (-1, "tab:blue")):
    charge_diffs = results[predicted_charge]["charge_diffs"]
    if charge_diffs:
        unique_diffs = sorted(set(charge_diffs))
        counts = [charge_diffs.count(d) for d in unique_diffs]
        total = sum(counts)
        fractions = [c / total for c in counts]
        ax3.plot(unique_diffs, fractions, "o-", linewidth=2, markersize=8,
                 label=f"Predicted charge {predicted_charge:+d} (n={len(charge_diffs)})", color=color)
ax3.set(xlabel="Charge of closest π± - Generator-level charge", ylabel="Fraction")
ax3.set_xticks(sorted(set([-2, -1, 0, 1, 2] + [d for diffs in results.values() for d in diffs["charge_diffs"]])))
ax3.legend()

ax4 = axes[3]
for predicted_charge, color in ((1, "tab:orange"), (-1, "tab:blue")):
    q_diff_values = results[predicted_charge]["q_diff"]
    if q_diff_values:
        # q_diff should be integers: -2, -1, 0, 1, 2
        unique_vals = sorted(set(q_diff_values))
        counts = [q_diff_values.count(v) for v in unique_vals]
        total = sum(counts)
        fractions = [c / total for c in counts]
        ax4.plot(unique_vals, fractions, "o-", linewidth=2, markersize=8,
                 label=f"Predicted charge {predicted_charge:+d} (n={len(q_diff_values)})", color=color)

ax4.set(xlabel="q_estimated - q_true", 
        ylabel="Fraction")
ax4.set_xticks(sorted(set([-2, -1, 0, 1, 2] + [v for diffs in results.values() for v in diffs["q_diff"]])))
ax4.legend()

fig.suptitle("DM 10, reconstructed PDGs [211, -211, 130], exactly 2 charged + 1 neutral")
plt.show()
plt.close()

In [ ]:
signal_modes = signal_labels
signal_truth = signal_true_dm.to_numpy()
signal_prediction = predicted_dm.iloc[:len(df)].to_numpy()
truth_signal_mask = signal_targets >= 0
mode_colors = dict(zip(signal_modes, plt.get_cmap('tab20', len(signal_modes))(
    np.arange(len(signal_modes)))))
background_colors = {'qq': 'black', 'mm': 'tab:olive', 'bhabha': 'tab:brown'}
momentum_cuts = np.linspace(0, 10, 21)


def momentum(frame, field):
    return frame[field].map(lambda p4: p4['pt'] * np.cosh(p4['eta'])).to_numpy()


def mode_metrics(truth, prediction, selected, denominator_sample=None):
    if denominator_sample is None:
        denominator_sample = selected
    efficiencies, precisions, f1_scores = [], [], []
    for mode in signal_modes:
        true_count = np.count_nonzero(denominator_sample & (truth == mode))
        pred_count = np.count_nonzero(denominator_sample & (prediction == mode))
        true_positive = np.count_nonzero(selected & (truth == mode) & (prediction == mode))
        efficiencies.append(true_positive / true_count if true_count else np.nan)
        precisions.append(true_positive / pred_count if pred_count else np.nan)
        f1_scores.append(2 * true_positive / (true_count + pred_count)
                         if true_count + pred_count else np.nan)
    return np.array([efficiencies, precisions, f1_scores])


def plot_mode_curves(x_values, values, background_values, xlabel, xlim, ymin, ylog=False):
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
    for metric_index, ax in enumerate(axes):
        for mode_index, mode in enumerate(signal_modes):
            ax.plot(x_values, values[:, metric_index, mode_index],
                    label=str(mode), color=mode_colors[mode])
        if metric_index == 0:
            for sample, label in background_samples.items():
                ax.plot(x_values, background_values[sample], label=label,
                        color=background_colors[sample], linestyle='--')
        ax.set(xlabel=xlabel, ylabel=('Efficiency', 'Precision', 'F1 score')[metric_index],
               xlim=xlim, ylim=(ymin, 1))
        ax.grid(alpha=0.25)
        if ylog and ymin > 0:
            ax.set_yscale('log')
    handles, curve_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, curve_labels, loc='center left', bbox_to_anchor=(0.82, 0.5))
    fig.subplots_adjust(left=0.055, right=0.80, bottom=0.16, wspace=0.3)
    plt.show()
    plt.close(fig)


background_class_indices = {
    sample: class_columns.index(f'tau_id_prob_{name}')
    for name, samples in background_groups.items() for sample in samples
}
background_predictions = {
    sample: (frame[class_columns].to_numpy() * tau_id_weights.to_numpy()).argmax(axis=1)
    for sample, frame in background_frames.items()
}


def plot_momentum_scan(field, xlabel):
    signal_momentum = momentum(df, field)
    background_momentum = {
        sample: momentum(frame, field) for sample, frame in background_frames.items()
    }
    momentum_values = []
    background_momentum_efficiencies = {sample: [] for sample in background_samples}
    for minimum_momentum in momentum_cuts:
        selected = truth_signal_mask & (signal_momentum >= minimum_momentum)
        momentum_values.append(mode_metrics(signal_truth, signal_prediction, selected))
        for sample in background_samples:
            selected_background = background_momentum[sample] >= minimum_momentum
            correct = (background_predictions[sample][selected_background]
                       == background_class_indices[sample])
            background_momentum_efficiencies[sample].append(
                np.mean(correct) if correct.size else np.nan
            )
    plot_mode_curves(momentum_cuts, np.array(momentum_values),
                     background_momentum_efficiencies, xlabel, (0, 10), 0.6)

In [ ]:
plot_momentum_scan('gen_jet_p4', 'Minimum generator jet momentum p [GeV]')

In [ ]:
plot_momentum_scan('reco_jet_p4', 'Minimum reconstructed jet momentum p [GeV]')

In [ ]:
signal_scores = df[class_columns].to_numpy() * tau_id_weights.to_numpy()
signal_class_indices = [class_columns.index(f'tau_id_prob_{name}') for name in signal_classes]
background_indices = [class_columns.index(f'tau_id_prob_{name}') for name in background_classes]
signal_margin = (signal_scores[:, signal_class_indices].max(axis=1)
                 / np.maximum(signal_scores[:, background_indices].max(axis=1), 1e-12))
best_signal_class = np.array(signal_classes)[
    signal_scores[:, signal_class_indices].argmax(axis=1)
]
signal_candidate_mode = np.array([
    predict_dm(classes, name) for classes, name in zip(
        df['pred_tau_daughter_meson_classes'], best_signal_class
    )
], dtype=object)
background_margins = {}
for sample, frame in background_frames.items():
    scores = frame[class_columns].to_numpy() * tau_id_weights.to_numpy()
    background_margins[sample] = (scores[:, signal_class_indices].max(axis=1)
                                  / np.maximum(scores[:, background_indices].max(axis=1), 1e-12))

thresholds = np.quantile(signal_margin[truth_signal_mask], np.linspace(0.2, 0, 25))
total_efficiencies = []
efficiency_values = []
background_cut_efficiencies = {sample: [] for sample in background_samples}
for threshold in thresholds:
    accepted_signal = signal_margin >= threshold
    total_efficiencies.append(np.mean(accepted_signal[truth_signal_mask]))
    efficiency_values.append(mode_metrics(signal_truth, signal_candidate_mode,
                                           truth_signal_mask & accepted_signal,
                                           denominator_sample=truth_signal_mask))
    for sample in background_samples:
        selected_background = background_margins[sample] >= threshold
        correct = (background_predictions[sample][selected_background]
                   == background_class_indices[sample])
        background_cut_efficiencies[sample].append(np.mean(correct) if correct.size else np.nan)

plot_mode_curves(np.array(total_efficiencies), np.array(efficiency_values),
                 background_cut_efficiencies, 'Minimum signal efficiency (full sample)', (0.8, 1), 0)

In [ ]:
import awkward as ak
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
from mltau.tools.general import reinitialize_p4

reconstruction_fields = {
    'Visible generator tau': ('gen_jet_tau_p4', False),
    #'Sum of generator daughters': ('gen_jet_tau_vis_daughter_p4s', True),
    'Reconstructed jet': ('reco_jet_p4', False),
    'Sum of regressed daughters': ('pred_tau_daughter_p4s', True),
}
reference_level = 'Visible generator tau'
response_levels = ('Reconstructed jet', 'Sum of regressed daughters')
selection_labels = {'A': 'A: Generator DM', 'B': 'B: Predicted DM', 'C': 'C: Both'}
reconstruction_modes = {
    'DM 0: Single pion': (0,),
    'DM 1: Rho': (1,),
    'DM 2': (2,),
    'DM 10': (10,),
    'a1: DM 2 or 10': (2, 10),
}
kinematic_bins = {
    'p [GeV]': np.linspace(0, 46, 47),
    'theta [deg]': np.linspace(10, 170, 41),
    'phi [rad]': np.linspace(-np.pi, np.pi, 26),
    'm [GeV]': np.linspace(0, 2.5, 51),
    'pT [GeV]': np.linspace(0, 46, 47),
    'E [GeV]': np.linspace(0, 46, 47),
    'eta': np.linspace(-2.5, 2.5, 51),
}
residual_units = {'theta [deg]': ' [deg]', 'phi [rad]': ' [rad]', 'eta': ''}
reconstruction_plot_dpi = 200


def reconstruction_quantities(series, sum_daughters):
    records = ak.Array([value.tolist() if isinstance(value, np.ndarray) else value for value in series])
    if not ak.fields(records):
        return {quantity: np.full(len(series), np.nan) for quantity in kinematic_bins}
    vectors = reinitialize_p4(records)
    if sum_daughters:
        vectors = ak.sum(vectors, axis=1, mask_identity=True)
    quantities = (
        vectors.p, np.degrees(vectors.theta), (vectors.phi + np.pi) % (2 * np.pi) - np.pi,
        vectors.mass, vectors.pt, vectors.energy, vectors.eta,
    )
    return {
        name: ak.to_numpy(ak.fill_none(values, np.nan))
        for name, values in zip(kinematic_bins, quantities)
    }


def divide_bins(numerator, denominator):
    return np.divide(numerator, denominator, out=np.full(denominator.shape, np.nan), where=denominator > 0)


def reconstruction_residual(values, reference, quantity):
    valid = np.isfinite(values) & np.isfinite(reference)
    if quantity not in residual_units:
        return divide_bins(np.where(valid, values, np.nan), np.where(reference > 1e-8, reference, np.nan))
    residual = np.full(len(reference), np.nan)
    residual[valid] = values[valid] - reference[valid]
    if quantity == 'phi [rad]':
        residual = (residual + np.pi) % (2 * np.pi) - np.pi
    return residual


def iqr_by_bin(reference, residual, edges):
    valid = np.isfinite(residual) & (reference >= edges[0]) & (reference <= edges[-1])
    bin_indices = np.minimum(np.searchsorted(edges, reference[valid], side='right') - 1, len(edges) - 2)
    order = np.argsort(bin_indices)
    residuals = residual[valid][order]
    boundaries = np.searchsorted(bin_indices[order], np.arange(len(edges)))
    return np.array([
        np.diff(np.percentile(residuals[start:stop], [25, 75]))[0] if stop - start >= 2 else np.nan
        for start, stop in zip(boundaries[:-1], boundaries[1:])
    ])


def reconstruction_curves(quantity, edges, selections, quantities, residuals):
    curves = {}
    bin_counts = {}
    for selection, mask in selections.items():
        reference = quantities[reference_level][quantity][mask]
        reference_counts = np.histogram(reference[np.isfinite(reference)], bins=edges)[0]
        for level in reconstruction_fields:
            values = quantities[level][quantity][mask]
            counts = np.histogram(values[np.isfinite(values)], bins=edges)[0]
            bin_counts[level, selection, 'Distribution'] = counts
            if level in response_levels:
                valid_scale = np.isfinite(reference) & np.isfinite(values)
                migration_counts = np.histogram2d(
                    reference[valid_scale], values[valid_scale], bins=(edges, edges),
                )[0]
                same_bin_counts = migration_counts.diagonal()

                curves[level, selection, 'Scale'] = divide_bins(same_bin_counts, reference_counts)
                bin_counts[level, selection, 'Scale'] = same_bin_counts
                valid_pairs = np.isfinite(reference) & np.isfinite(residuals[level][quantity][mask])
                bin_counts[level, selection, 'Response'] = np.histogram(reference[valid_pairs], bins=edges)[0]
            curves[level, selection, 'Distribution'] = (
                counts / (counts.sum() * np.diff(edges)) if counts.sum() else np.full(counts.shape, np.nan)
            )
            if level in response_levels:
                curves[level, selection, 'Scale'] = divide_bins(counts, reference_counts)
                curves[level, selection, 'Response'] = iqr_by_bin(reference, residuals[level][quantity][mask], edges)
    return curves, bin_counts


def ratio_ceiling(curves):
    values = np.concatenate([curve[np.isfinite(curve)] for curve in curves]) if curves else np.array([])
    return max(2.0, 1.15 * np.percentile(values, 95)) if values.size else 2.0


def draw_reconstruction_curve(axis, edges, values, style, ceiling=None):
    shown = np.where(values <= ceiling, values, np.nan) if ceiling is not None else values
    axis.stairs(shown, edges, baseline=None, **style)
    if ceiling is not None:
        overflow = values > ceiling
        if overflow.any():
            centers = (edges[:-1] + edges[1:]) / 2
            axis.plot(centers[overflow], np.full(overflow.sum(), 0.97 * ceiling),
                      '^', color=style['color'], markersize=3, zorder=style['zorder'])


def plot_reconstruction_comparison(mode_label, selections, quantities, residuals, compare_levels):
    metrics = ('Distribution', 'Scale', 'Response')
    overlay = reconstruction_fields if compare_levels else selection_labels
    panels = [
        (panel, metric)
        for panel in (selection_labels if compare_levels else reconstruction_fields)
        for metric in metrics
        if compare_levels or metric == 'Distribution' or panel in response_levels
    ]
    colors = dict(zip(overlay, plt.get_cmap('tab10').colors))
    line_styles = ('-', '-', '--') if compare_levels else ('-', '--', '-.')
    styles = {
        key: dict(color=colors[key], linestyle=linestyle, linewidth=1.15,
                  zorder=5 if key == reference_level else 3)
        for key, linestyle in zip(overlay, line_styles)
    }
    figure = plt.figure(figsize=(3.0 * len(panels), 3.0 * len(kinematic_bins)), dpi=reconstruction_plot_dpi)
    grid = figure.add_gridspec(len(kinematic_bins), len(panels), left=0.045, right=0.99,
                              bottom=0.07, top=0.94, wspace=0.36, hspace=0.62)
    comparison = 'Compare reconstruction levels' if compare_levels else 'Compare selections'
    figure.suptitle(f'{mode_label} | {comparison}', fontsize=16, y=0.985)
    for row_index, (quantity, edges) in enumerate(kinematic_bins.items()):
        curves, bin_counts = reconstruction_curves(quantity, edges, selections, quantities, residuals)
        shared_axes = {}
        for column_index, (panel, metric) in enumerate(panels):
            pair = grid[row_index, column_index].subgridspec(2, 1, height_ratios=(4, 1), hspace=0)
            #axis = figure.add_subplot(pair[0], sharey=None if metric == 'Scale' else shared_axes.get(metric))
            axis = figure.add_subplot(pair[0])
            ratio_axis = figure.add_subplot(pair[1], sharex=axis)
            shared_axes.setdefault(metric, axis)
            panel_label = selection_labels[panel] if compare_levels else panel
            if row_index == 0:
                axis.set_title(f'{panel_label}\n{metric}', fontsize=10)
            axis.tick_params(axis='x', labelbottom=False, bottom=False)
            axis.tick_params(axis='y', labelsize=8)
            ratio_axis.tick_params(axis='y', left=True, labelleft=True, labelright=False, labelsize=7)
            axis.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='lower'))
            ratio_axis.yaxis.set_major_locator(MaxNLocator(nbins=3, prune='upper'))
            xlabel = f'Generator {quantity}' if metric in ('Scale', 'Response') else quantity
            ratio_axis.set(xlim=(edges[0], edges[-1]), xlabel=xlabel)
            baseline = (reference_level if metric == 'Distribution' else response_levels[0]) if compare_levels else 'A'
            if axis is shared_axes[metric]:
                axis.set_ylabel({'Distribution': 'Probability density', 'Scale': 'Reco / gen count',
                                 'Response': f'IQR (75-25){residual_units.get(quantity, "")}'}[metric],
                                 fontsize=8)
            #ratio_axis.set_ylabel((' / Gen' if baseline == reference_level else ' / Jet' if compare_levels else ' / A').strip())
            axis.grid(alpha=0.2)
            ratio_axis.grid(alpha=0.2)
            ratio_axis.axhline(1, color='0.45', linewidth=0.8)
            panel_curves = {}
            for key in overlay:
                level, selection = (key, panel) if compare_levels else (panel, key)
                curve = curves.get((level, selection, metric))
                if curve is not None:
                    panel_curves[key] = curve
                    counts = bin_counts[level, selection, metric]
                    error = divide_bins(curve, np.sqrt(counts))
                    axis.stairs(
                        curve + error, edges,
                        baseline=np.maximum(0, curve - error),
                        fill=True, color=styles[key]['color'], alpha=0.18,
                        linewidth=0, zorder=styles[key]['zorder'] - 1,
                    )
                    draw_reconstruction_curve(axis, edges, curve, styles[key])
            if metric != 'Scale':
                axis.set_ylim(bottom=0)
            denominator = panel_curves[baseline]
            ratios = {key: divide_bins(values, denominator) for key, values in panel_curves.items() if key != baseline}
            ceiling = ratio_ceiling(list(ratios.values()))
            ratio_axis.set_ylim(0, ceiling)
            for key, ratio in ratios.items():
                draw_reconstruction_curve(ratio_axis, edges, ratio, styles[key], ceiling)
    handles = [
        Line2D([], [], label=key if compare_levels else selection_labels[key],
               **{name: value for name, value in styles[key].items() if name != 'zorder'})
        for key in overlay
    ]
    figure.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.021), ncol=len(handles))
    figure.text(0.5, 0.012, 'Triangles mark bins above the shown range; ratios with zero denominators are omitted.', ha='center')
    plt.show()
    plt.close(figure)

In [ ]:
reconstruction_values = {
    level: reconstruction_quantities(df[field], sum_daughters)
    for level, (field, sum_daughters) in reconstruction_fields.items()
}

reconstruction_residuals = {
    level: {
        quantity: reconstruction_residual(
            reconstruction_values[level][quantity],
            reconstruction_values[reference_level][quantity],
            quantity,
        )
        for quantity in kinematic_bins
    }
    for level in response_levels
}

reconstruction_cut_settings = (False, True)
minimum_reco_p_by_predicted_dm = {0: 8, 1: 10, 2: 20, 10: 15}

reconstruction_truth_dm = signal_true_dm.to_numpy()
reconstruction_predicted_dm = predicted_dm.iloc[:len(df)].to_numpy()
#reco_jet_momentum = reconstruction_values['Reconstructed jet']['p [GeV]']
reco_jet_momentum = reconstruction_values['Sum of regressed daughters']['p [GeV]']

uncut_mask = np.ones(len(df), dtype=bool)
momentum_cut_mask = uncut_mask.copy()
for predicted_mode, minimum_p in minimum_reco_p_by_predicted_dm.items():
    matched = reconstruction_predicted_dm == predicted_mode
    momentum = reco_jet_momentum[matched]
    momentum_cut_mask[matched] = np.isfinite(momentum) & (momentum > minimum_p)

for apply_cuts in reconstruction_cut_settings:
    accepted = momentum_cut_mask if apply_cuts else uncut_mask
    cut_label = 'DM-dependent reco p cuts' if apply_cuts else 'No momentum cuts'

    for mode_label, modes in reconstruction_modes.items():
        true_mode = np.isin(reconstruction_truth_dm, modes)
        predicted_mode = np.isin(reconstruction_predicted_dm, modes)
        mode_selections = {
            'A': accepted & true_mode,
            'B': accepted & predicted_mode,
            'C': accepted & true_mode & predicted_mode,
        }
        plot_label = f'{mode_label} | {cut_label}'
        selection_counts = ', '.join(
            f'{name} = {np.count_nonzero(mask):,}'
            for name, mask in mode_selections.items()
        )
        print(f'{plot_label} (selected jets: {selection_counts})')

        for compare_levels in (True, False):
            plot_reconstruction_comparison(
                plot_label, mode_selections, reconstruction_values,
                reconstruction_residuals, compare_levels,
            )

In [ ]:
def plot_reconstruction_migrations(title, selections):
    panels = [
        (selection, level)
        for selection in selections
        for level in response_levels
    ]
    figure, axes = plt.subplots(
        len(kinematic_bins), len(panels), squeeze=False,
        figsize=(3 * len(panels), 2.65 * len(kinematic_bins)),
        dpi=reconstruction_plot_dpi, layout='constrained',
    )
    cmap = plt.get_cmap('viridis').with_extremes(bad='white')
    figure.suptitle(f'{title} | Generator-to-reconstruction correlations')

    for row, (quantity, edges) in enumerate(kinematic_bins.items()):
        for column, (selection, level) in enumerate(panels):
            axis = axes[row, column]
            mask = selections[selection]
            generator = reconstruction_values[reference_level][quantity][mask]
            reconstructed = reconstruction_values[level][quantity][mask]
            valid = np.isfinite(generator) & np.isfinite(reconstructed)

            counts = np.histogram2d(
                generator[valid], reconstructed[valid], bins=(edges, edges),
            )[0].T
            totals = counts.sum(axis=0, keepdims=True)
            fractions = np.divide(
                counts, totals, out=np.zeros_like(counts), where=totals > 0,
            )
            mesh = axis.pcolormesh(
                edges, edges, np.ma.masked_where(counts == 0, fractions),
                shading='flat', cmap=cmap, vmin=0, vmax=1,
            )
            axis.set(xlabel=f'Generator {quantity}',
                     xlim=(edges[0], edges[-1]), ylim=(edges[0], edges[-1]))
            if column == 0:
                axis.set_ylabel(f'Reconstructed {quantity}')
            if row == 0:
                axis.set_title(f'{selection_labels[selection]}\n{level}', fontsize=9)

    figure.colorbar(mesh, ax=axes[0, :], label='Fraction within generator bin', shrink=0.8)
    plt.show()
    plt.close(figure)


for cuts_enabled in reconstruction_cut_settings:
    accepted_jets = momentum_cut_mask if cuts_enabled else uncut_mask
    cut_status = 'Momentum cuts enabled' if cuts_enabled else 'No momentum cuts'
    for decay_label, decay_modes in reconstruction_modes.items():
        truth_matches = np.isin(reconstruction_truth_dm, decay_modes)
        prediction_matches = np.isin(reconstruction_predicted_dm, decay_modes)
        selections = {
            'A': accepted_jets & truth_matches,
            'B': accepted_jets & prediction_matches,
            'C': accepted_jets & truth_matches & prediction_matches,
        }
        plot_reconstruction_migrations(f'{decay_label} | {cut_status}', selections)